# 04 — Evaluation: Measuring What Fine-Tuning Changed

**Project**: LLM Fine-Tuning with QLoRA  
**Notebook**: 4 of 6  
**GPU required**: ✅ Yes — T4  
**Runtime**: ~20 minutes

---

## What This Notebook Does

Now that training is complete, we measure the impact.
Three evaluation methods are used together — because no single metric tells the full story.

| Method | What it measures | Limitation |
|---|---|---|
| **Qualitative** | Output quality, format, instruction following | Subjective, small sample |
| **ROUGE scores** | Word-level overlap with reference answers | Misses paraphrases |
| **Loss curve** | Training health, overfitting | Only measures training signal |

## The Central Question
> Did fine-tuning on 2,000 Alpaca examples meaningfully improve TinyLlama's
> instruction-following ability — and by how much?

---
## ⚙️ Cell 1: Setup

In [ ]:
!pip install -q transformers peft datasets bitsandbytes accelerate evaluate rouge-score matplotlib
print("✅ Dependencies installed")

In [ ]:
import os, sys, json

GITHUB_USERNAME = "YOUR_USERNAME"   # ← change this
REPO_NAME = "llm-finetuning-peft"

if not os.path.exists(REPO_NAME):
    !git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
else:
    !cd {REPO_NAME} && git pull

sys.path.insert(0, f"/content/{REPO_NAME}/src")
os.chdir(f"/content/{REPO_NAME}")
os.makedirs("results", exist_ok=True)
print(f"✅ Ready. CWD: {os.getcwd()}")

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ GPU required. Enable T4 GPU in runtime settings.")
print(f"✅ GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

---
## 🤖 Cell 2: Load the Fine-Tuned Model

We load the base model + the LoRA adapter we saved in notebook 03.
This is exactly how anyone would use your fine-tuned model.

In [ ]:
from model_utils import load_tokenizer, load_base_model
from peft import PeftModel

MODEL_NAME   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_PATH = "outputs/qlora-adapter-r8"
LORA_R       = 8

print("📥 Loading base model...")
tokenizer  = load_tokenizer(MODEL_NAME)
base_model = load_base_model(MODEL_NAME)

print(f"\n🔌 Loading LoRA adapter from: {ADAPTER_PATH}")
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
ft_model.eval()

print("\n✅ Fine-tuned model loaded and ready!")
print(f"   Device: {next(ft_model.parameters()).device}")

---
## 🔍 Cell 3: Qualitative Evaluation — Before vs After

We load the saved baseline outputs from notebook 02 and compare them
to what the fine-tuned model generates on the exact same prompts.

In [ ]:
from evaluate import QUALITATIVE_PROMPTS, generate_response

# Load saved baseline outputs from notebook 02
BASELINE_PATH = "results/baseline_outputs.json"

if os.path.exists(BASELINE_PATH):
    with open(BASELINE_PATH) as f:
        baseline_data = {item["label"]: item["base_response"] for item in json.load(f)}
    print(f"✅ Loaded baseline outputs from: {BASELINE_PATH}")
else:
    print("⚠️  baseline_outputs.json not found — run notebook 02 first, or baseline will be empty")
    baseline_data = {}

comparison_results = []

print("\n" + "=" * 70)
print("  QUALITATIVE COMPARISON — BASE MODEL vs FINE-TUNED")
print("=" * 70)

for item in QUALITATIVE_PROMPTS:
    label  = item["label"]
    prompt = item["prompt"]
    instruction = prompt.split("### Instruction:")[1].split("### Response:")[0].strip()

    print(f"\n📌 [{label}]")
    print(f"Instruction: {instruction}")
    print("-" * 70)

    # Base model response (from saved file or placeholder)
    base_resp = baseline_data.get(label, "[Run notebook 02 to generate baseline]")
    print(f"🔵 BASE MODEL:\n{base_resp}")

    print()

    # Fine-tuned model response
    ft_resp = generate_response(ft_model, tokenizer, prompt, max_new_tokens=200)
    print(f"🟣 FINE-TUNED:\n{ft_resp}")
    print("=" * 70)

    comparison_results.append({
        "label": label,
        "instruction": instruction,
        "base_response": base_resp,
        "finetuned_response": ft_resp,
    })

# Save comparison
with open("results/qualitative_comparison.json", "w", encoding="utf-8") as f:
    json.dump(comparison_results, f, indent=2, ensure_ascii=False)
print("\n💾 Qualitative comparison saved to results/qualitative_comparison.json")

In [ ]:
# ── What to look for in the comparison ───────────────────────────────────
print("🔍 Evaluation Checklist — review each response pair above:")
print()
print("FORMAT:")
print("  □ Does the fine-tuned model use numbered lists more consistently?")
print("  □ Does it stop generating at an appropriate length?")
print("  □ Does it avoid repeating phrases or looping?")
print()
print("CONTENT:")
print("  □ Does the response actually answer the instruction?")
print("  □ Is the content more concise and on-topic?")
print("  □ Is the quality noticeably better?")
print()
print("💡 Remember: fine-tuning teaches STYLE not new knowledge.")
print("   The model doesn't learn new facts — it learns how to present them.")

---
## 📊 Cell 4: ROUGE Score Evaluation

We now compute ROUGE on 100 test examples — comparing generated text to reference answers.

**How ROUGE works:**
- **ROUGE-1**: single word overlap (unigrams)
- **ROUGE-2**: word pair overlap (bigrams)  
- **ROUGE-L**: longest common subsequence

All scores are between 0 and 1. Higher = more overlap with reference.

In [ ]:
from data_utils import load_alpaca_dataset
from evaluate import compute_rouge_scores
import evaluate as hf_evaluate

_, test_dataset = load_alpaca_dataset(train_size=2000, test_size=200, seed=42)

NUM_EVAL_SAMPLES = 100  # Evaluate on 100 samples — takes ~10 minutes

print(f"📊 Computing ROUGE on {NUM_EVAL_SAMPLES} test samples...")
print("   Each sample: model generates a response → compare to reference")
print("   This takes ~10 minutes — each token is generated one at a time\n")

# Evaluate fine-tuned model
ft_rouge = compute_rouge_scores(ft_model, tokenizer, test_dataset, num_samples=NUM_EVAL_SAMPLES)

print("\n" + "=" * 50)
print("  ROUGE SCORES — FINE-TUNED MODEL")
print("=" * 50)
print(f"  ROUGE-1 : {ft_rouge['rouge1']:.4f}")
print(f"  ROUGE-2 : {ft_rouge['rouge2']:.4f}")
print(f"  ROUGE-L : {ft_rouge['rougeL']:.4f}")

In [ ]:
# ── Also evaluate the base model for comparison ────────────────────────────
# We need to load the base model separately (no adapter)
from model_utils import load_base_model as load_base

print("📊 Computing ROUGE on BASE model (no fine-tuning) for comparison...")

# Load a clean base model (without LoRA adapter)
base_only = load_base(MODEL_NAME)
base_only.eval()

base_rouge = compute_rouge_scores(base_only, tokenizer, test_dataset, num_samples=NUM_EVAL_SAMPLES)

# Free base model from memory — we don't need it anymore
del base_only
torch.cuda.empty_cache()

print("\n" + "=" * 50)
print("  ROUGE COMPARISON")
print("=" * 50)
print(f"  {'Metric':<12} {'Base':>10} {'Fine-Tuned':>12} {'Improvement':>13}")
print("  " + "-" * 48)
for metric, label in [("rouge1", "ROUGE-1"), ("rouge2", "ROUGE-2"), ("rougeL", "ROUGE-L")]:
    base_val = base_rouge[metric]
    ft_val   = ft_rouge[metric]
    pct = (ft_val - base_val) / max(base_val, 1e-8) * 100
    arrow = "▲" if ft_val > base_val else "▼"
    print(f"  {label:<12} {base_val:>10.4f} {ft_val:>12.4f} {arrow}{abs(pct):>10.1f}%")

---
## 📈 Cell 5: Visualization — ROUGE Comparison Chart

In [ ]:
from evaluate import plot_rouge_comparison, save_evaluation_summary

# Generate and save the bar chart
plot_rouge_comparison(
    base_scores=base_rouge,
    finetuned_scores=ft_rouge,
    save_path="results/rouge_scores_comparison.png"
)

# Save structured JSON with all metrics
save_evaluation_summary(
    base_rouge=base_rouge,
    finetuned_rouge=ft_rouge,
    lora_rank=LORA_R,
    save_path="results/evaluation_results.json"
)

print("\n📌 These numbers go into your README results table!")
with open("results/evaluation_results.json") as f:
    eval_data = json.load(f)

print("\nREADME table values:")
print(f"  | ROUGE-1 | {eval_data['base_model']['rouge1']} | {eval_data['finetuned_model']['rouge1']} |")
print(f"  | ROUGE-2 | {eval_data['base_model']['rouge2']} | {eval_data['finetuned_model']['rouge2']} |")
print(f"  | ROUGE-L | {eval_data['base_model']['rougeL']} | {eval_data['finetuned_model']['rougeL']} |")

---
## 📉 Cell 6: Display the Training Loss Curve

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt

loss_curve_path = "results/training_loss_curve.png"

if os.path.exists(loss_curve_path):
    print("📉 Training Loss Curve (generated in notebook 03):")
    display(Image(loss_curve_path))
else:
    print("⚠️  Loss curve not found. Run notebook 03 first.")

In [ ]:
# ── Interpret the loss curve ───────────────────────────────────────────────
if os.path.exists("outputs/qlora-adapter-r8/training_metrics.yaml"):
    import yaml
    with open("outputs/qlora-adapter-r8/training_metrics.yaml") as f:
        metrics = yaml.safe_load(f)

    print("📊 Training Metrics Summary:")
    print("-" * 45)
    print(f"   Initial loss         : {metrics.get('initial_loss', 'N/A')}")
    print(f"   Final loss           : {metrics.get('final_loss', 'N/A')}")
    print(f"   Loss reduction       : {metrics.get('loss_reduction_pct', 'N/A')}%")
    print(f"   Total steps          : {metrics.get('total_steps', 'N/A')}")
    print(f"   Training time        : {metrics.get('training_time_minutes', 'N/A')} min")
    print()
    print("💡 Loss curve interpretation:")
    print("   - Consistent downward trend = healthy training ✅")
    print("   - Plateau before epoch 2 = could train longer")
    print("   - Rising tail = potential overfitting on 2K samples")

---
## 📤 Cell 7: Commit All Results

In [ ]:
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"
!git add results/
!git commit -m "results: add ROUGE scores, comparison chart, qualitative evaluation"
!git push
print("✅ All evaluation results committed to GitHub!")

---
## ✅ Summary

| Step | Output |
|---|---|
| Qualitative | Side-by-side base vs fine-tuned on 5 prompts |
| ROUGE scores | Computed on 100 test samples → concrete numbers |
| Loss curve | Visual confirmation of healthy training |
| Results saved | `evaluation_results.json`, `rouge_scores_comparison.png` |

### Key Evaluation Insight
ROUGE improvement shows quantitative gain. But qualitative review reveals *how* the model
improved — better format adherence, cleaner responses, correct stopping behavior.
Both matter. Neither alone is sufficient.

---
### ▶️ Next: `05_ablation_study.ipynb`
We compare LoRA rank `r=4` vs `r=8` vs `r=16` — the experiment that makes this project stand out.